<a href="https://colab.research.google.com/github/veronicaluzzi/Data-Science-Cohort-20/blob/main/Project-5/NLP_part3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Natural Language Processing



This project will give you practical experience using Natural Language Processing techniques. This project is in three parts:
- in part 1) you will use a dataset in a CSV file
- in part 2) you will use the Wikipedia API to directly access content
on Wikipedia.
- in part 3) you will make your notebook interactive


### Part 3)


Make an interactive notebook where a user can choose or enter a name and the notebook displays the 10 closest individuals.

In addition to presenting the project slides, at the end of the presentation each student will demonstrate their code using a famous person suggested by the other students that exists in the DBpedia set.


In [ ]:
%%capture output
#install Wikipedia API
!pip3 install wikipedia-api

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from textblob import TextBlob
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

from ipywidgets import interact, widgets

import time

import wikipediaapi

pd.options.display.max_columns = 100

import nltk
# nltk.download('omw-1.4')
nltk.download('punkt_tab')
# nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
!curl -s https://ddc-datascience.s3.amazonaws.com/Projects/Project.5-NLP/Data/NLP.csv | wc -l

42786


In [ ]:
%%bash
curl -s https://ddc-datascience.s3.amazonaws.com/Projects/Project.5-NLP/Data/NLP.csv |
head -1 |
tr , '\n' |
cat -n


     1	URI
     2	name
     3	text


In [ ]:
%%bash
curl -s https://ddc-datascience.s3.amazonaws.com/Projects/Project.5-NLP/Data/NLP.csv |
head -2 |
tail -1 |
tr , '\n' |
cat -n


     1	<http://dbpedia.org/resource/Digby_Morrell>
     2	Digby Morrell
     3	digby morrell born 10 october 1979 is a former australian rules footballer who played with the kangaroos and carlton in the australian football league aflfrom western australia morrell played his early senior football for west perth his 44game senior career for the falcons spanned 19982000 and he was the clubs leading goalkicker in 2000 at the age of 21 morrell was recruited to the australian football league by the kangaroos football club with its third round selection in the 2001 afl rookie draft as a forward he twice kicked five goals during his time with the kangaroos the first was in a losing cause against sydney in 2002 and the other the following season in a drawn game against brisbaneafter the 2003 season morrell was traded along with david teague to the carlton football club in exchange for corey mckernan he played 32 games for the blues before being delisted at the end of 2005 he continued to play v

Part 1 code below

In [ ]:
# Use .iloc[0] to grab the text of the person of interest (Gary Griffin)
# This looks at the 42630th row, and extracts whatever is in the 'text' column
griffin_full_text = df["text"].iloc[42630]

# Create a TextBlob of his full text profile
full_blob = TextBlob(griffin_full_text)
full_blob


TextBlob("gary griffin born october 26 1951 in cincinnati ohio is an american musician best known for performing as a keyboardist and vocalist with the beach boys brian wilson jan and dean and the surf city allstarsgriffin grew up in cincinnati and attended miami university and the university of cincinnati college conservatory of music where he majored in piano and music theoryin 1977 griffin moved to los angeles where he was hired as an organist by the beach boys joining them for the recording of their warner bros album miu griffin toured and recorded with jan and dean throughout the 80s and 90s and has appeared on several television shows most notably general hospital and full house in 2000 griffin served as music director for the emmynominated television miniseries the beach boys an american family for abc he also appeared in two roles in the film which was produced by john stamos griffin has produced an eclectic roster of artists most notably micky dolenz of the monkees and politic

In [ ]:
size = len(full_blob)
size

1711

In [ ]:
# convert the text for Gary Griffin into a string to be able to vectorize it. full_blob is a textblob.
full_blob_string = [str(s) for s in full_blob.sentences]
full_blob_string

['gary griffin born october 26 1951 in cincinnati ohio is an american musician best known for performing as a keyboardist and vocalist with the beach boys brian wilson jan and dean and the surf city allstarsgriffin grew up in cincinnati and attended miami university and the university of cincinnati college conservatory of music where he majored in piano and music theoryin 1977 griffin moved to los angeles where he was hired as an organist by the beach boys joining them for the recording of their warner bros album miu griffin toured and recorded with jan and dean throughout the 80s and 90s and has appeared on several television shows most notably general hospital and full house in 2000 griffin served as music director for the emmynominated television miniseries the beach boys an american family for abc he also appeared in two roles in the film which was produced by john stamos griffin has produced an eclectic roster of artists most notably micky dolenz of the monkees and political satir

In [ ]:
size_1 = len(full_blob_string)
size_1

1

In [ ]:
#vectorizing full_blob_string
vectorizer = CountVectorizer(stop_words="english")
bow_griffin = vectorizer.fit_transform(full_blob_string)

In [ ]:
# to check the code above worked.
print("BoW Griffin Shape:", bow_griffin.shape)
print("Vocabulary Preview:", list(vectorizer.vocabulary_.keys())[:10])

BoW Griffin Shape: (1, 127)
Vocabulary Preview: ['gary', 'griffin', 'born', 'october', '26', '1951', 'cincinnati', 'ohio', 'american', 'musician']


In [ ]:
# Make sure there are no missing values in the text column
# (CountVectorizer will error out if it encounters a NaN/Float value)
df['text'] = df["text"].fillna("")

In [ ]:
# Initialize the CountVectorizer
# stop_words='english' removes common filler words like 'the', 'is', 'and'
vectorizer = CountVectorizer(stop_words="english")

In [ ]:
# Transform the entire text column straight into a Bag of Words matrix
# it works on all rows simultaneously!
bow_alltext = vectorizer.fit_transform(df["text"])

In [ ]:
# --- VERIFY THE OUTPUT ---
print("--- VECTORIZATION COMPLETE ---")
print(f"Total number of people (rows): {bow_alltext.shape[0]}")
print(f"Total number of unique words (columns): {bow_alltext.shape[1]}")
print(f"Type of matrix generated: {type(bow_alltext)}")

--- VECTORIZATION COMPLETE ---
Total number of people (rows): 42786
Total number of unique words (columns): 437190
Type of matrix generated: <class 'scipy.sparse._csr.csr_matrix'>


In [ ]:
# Transform BoW to TF-IDF
tfidf_transformer = TfidfTransformer()
tfidf_matrix = tfidf_transformer.fit_transform(bow_alltext)

# Fit your KNN model on the matrix
knn = NearestNeighbors(n_neighbors=11, metric="cosine")
knn.fit(tfidf_matrix)

NearestNeighbors(metric='cosine', n_neighbors=11)

In [ ]:
distances, indices = knn.kneighbors(
  X = tfidf_matrix[42630],#reference cell
  n_neighbors = 11,# what are the 10 more close to it
)

In [ ]:
indices

array([[42630,  2850, 33569,  5034,  1716,  4665, 36348, 10459, 26239,
        16970, 13115]])

In [ ]:
distances

array([[0.        , 0.66875993, 0.70739112, 0.72445141, 0.73112111,
        0.73421783, 0.75153767, 0.75912665, 0.7618706 , 0.78103172,
        0.80559001]])

In [ ]:
# 1.For this code to work, I need to run part 1 first. The code below is using the references df, knn, and code from cells above.
# 2. Use the @interact decorator to automatically generate the slider.
# When you move the slider, it selects an integer (row_index = 500). I choose a max = 1000 to be brief.
@interact(row_index=widgets.IntSlider(min=0, max=1000, step=1, value=0, description='Row Index:'))
# 3. Create a function to gather all the information I need for the slider
def show_closest_by_row(row_index):
    # Query the KNN model using the selected slider index
    # (Matches  the code above: distances, indices = knn.kneighbors(...))
    distances, indices = knn.kneighbors(tfidf_matrix[row_index], n_neighbors=11)

    neighbor_indices = indices[0]
    neighbor_distances = distances[0]

    # Create a temporary DataFrame to process and display results
    results_df = df.iloc[neighbor_indices].copy()
    results_df['Cosine Distance'] = neighbor_distances

    # Remove the reference individual themselves so only the 10 closest show
    results_df = results_df[results_df.index != row_index]

    # Clean up the dataframe so is human readable
    results_df.insert(0, 'Rank', range(1, 11))
    results_df = results_df[['Rank', 'name', 'Cosine Distance', 'URI']]
    results_df.set_index('Rank', inplace=True)

    # Add an f string to make clear who the target person is

    print(f"Target Person at row {row_index}: {df.loc[row_index, 'name']}\n")
    display(results_df)

interactive(children=(IntSlider(value=0, description='Row Index:', max=1000), Output()), _dom_classes=('widget…